# Rwanda Road Safety Intelligence System (RRSIS)
## HDFS + Apache Spark (PySpark DataFrame) Analytics Engine

### Executive Summary & Project Case Study
Road traffic accidents present a grave socio-economic and public health challenge in Rwanda. According to the **National Institute of Statistics of Rwanda (NISR) Statistical Yearbook 2024** (Table 14.2.6: *Road Accidents*), Rwanda recorded **9,995 total road accidents in 2023**, resulting in **761 fatal accidents**, **5,037 minor injury accidents**, and **3,935 property damage accidents** (figures sourced directly from Rwanda National Police records).

To transition from manual reporting to a data-driven road safety intelligence framework, this prototype **Rwanda Road Safety Intelligence System (RRSIS)** is built on **Apache Spark (PySpark DataFrame API)** with **Hadoop Distributed File System (HDFS)** as the distributed storage layer.

> **Dataset Transparency Disclosure**: In accordance with project requirements, the Kaggle Road Accident Dataset (307,973 records, 23 attributes) is utilized as an **enterprise surrogate dataset** for developing and validating the PySpark analytics engine. Official Rwandan macro-statistics from NISR are integrated for context, and a roadmap for ingesting actual Rwanda National Police (RNP) accident micro-data is provided.


## Task 1: HDFS + Spark Data Ingestion & Partition Diagnostics
Loading accident records directly from **HDFS** (or local surrogate path), inspecting DataFrame schema, verifying record/column dimensions, generating statistical summaries, and auditing distributed RDD partition layouts using `spark_partition_id()`.


In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize PySpark Session
spark = SparkSession.builder \
    .appName("Rwanda_Road_Safety_Intelligence_System") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# HDFS Dataset URI with local fallback for local development
HDFS_DATASET_PATH = "hdfs://localhost:9000/user/hadoop/rrsis/raw/road_accidents_23cols.csv"
LOCAL_DATASET_PATH = "e:/archive (7)/Road Accident Data.csv"

target_path = HDFS_DATASET_PATH
if not target_path.startswith("hdfs://") or not os.path.exists(LOCAL_DATASET_PATH):
    pass

# Read raw dataset into PySpark DataFrame
try:
    df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(HDFS_DATASET_PATH)
    print(f"[HDFS INGESTION SUCCESS] Loaded dataset from HDFS: {HDFS_DATASET_PATH}")
except Exception as e:
    print(f"[LOCAL SURROGATE FALLBACK] HDFS unavailable ({e}). Loading local dataset: {LOCAL_DATASET_PATH}")
    df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(LOCAL_DATASET_PATH)

# Standardize column headers (remove spaces and parentheses)
for col in df_raw.columns:
    clean_col = col.replace(" ", "_").replace("(", "").replace(")", "")
    df_raw = df_raw.withColumnRenamed(col, clean_col)

# Schema & Dimension Diagnostics
record_count = df_raw.count()
column_count = len(df_raw.columns)
print("==================================================================")
print(f" TOTAL INGESTED ACCIDENT RECORDS : {record_count:,}")
print(f" TOTAL ATTRIBUTES / COLUMNS      : {column_count}")
print("==================================================================")

# Display PySpark Schema
print("
--- PySpark DataFrame Schema ---")
df_raw.printSchema()

# Partition Diagnostics
num_partitions = df_raw.rdd.getNumPartitions()
print(f"
Total Spark RDD Partitions: {num_partitions}")

# RDD Partition ID distribution via spark_partition_id()
df_partitioned = df_raw.withColumn("partition_id", F.spark_partition_id())
print("
Partition Record Allocation:")
df_partitioned.groupBy("partition_id").count().orderBy("partition_id").show()

# Sample Records
print("Sample Records (First 2 Rows):")
df_raw.show(2, truncate=False)


## Task 2: Data Quality Engineering & Sanitization (Audit BEFORE Cleaning)

### Mandatory Data Quality Audit BEFORE Cleaning
Before applying any transformations, we perform an explicit audit on `df_raw` to identify data quality issues:
1. **Missing / Null Values**: Count of `null`, empty string `""`, `"None"`, and `"NULL"` per column.
2. **Duplicate Records**: Exact duplicate row count.
3. **Inappropriate Data Types & Range Outliers**: Non-positive speed limits (`Speed_limit <= 0`), invalid zero coordinates (`Latitude == 0` or `Longitude == 0`).
4. **Typographical Errors**: Inconsistent casing and typos (e.g. `"Fetal"` instead of `"Fatal"`).
5. **Analytical Impact**: Documenting how removing vs. imputing missing fields preserves statistical sample integrity.


In [1]:
# ------------------------------------------------------------------
# STEP 1: AUDIT DATA QUALITY ISSUES BEFORE CLEANING
# ------------------------------------------------------------------
print("==================================================================")
print("       TASK 2: DATA QUALITY AUDIT (BEFORE CLEANING)")
print("==================================================================")

# 1. Audit missing/null count per column
null_audit_exprs = [
    F.count(
        F.when(
            F.col(c).isNull() | 
            (F.col(c).cast("string") == "") | 
            (F.col(c).cast("string") == "None") | 
            (F.col(c).cast("string") == "NULL"), 
            c
        )
    ).alias(c) for c in df_raw.columns
]
df_null_audit = df_raw.select(null_audit_exprs)
print("
[1] Missing / Null Count per Column (BEFORE Cleaning):")
df_null_audit.show(vertical=True)

# 2. Audit duplicate rows count
raw_count = df_raw.count()
dup_count = raw_count - df_raw.dropDuplicates().count()
print(f"[2] Duplicate Row Count (BEFORE Cleaning): {dup_count:,} ({(dup_count/raw_count)*100:.2f}%)")

# 3. Audit invalid numeric ranges
invalid_speed_count = df_raw.filter((F.col("Speed_limit").isNull()) | (F.col("Speed_limit") <= 0)).count()
zero_coords_count = df_raw.filter((F.col("Latitude") == 0) | (F.col("Longitude") == 0) | (F.col("Latitude").isNull()) | (F.col("Longitude").isNull())).count()
print(f"[3] Invalid / Non-Positive Speed Limits: {invalid_speed_count:,}")
print(f"[4] Zero or Null Coordinates Count:      {zero_coords_count:,}")

# 4. Audit distinct values in Accident_Severity
print("
[5] Distinct Accident_Severity Values (BEFORE Cleaning):")
df_raw.groupBy("Accident_Severity").count().show()

# ------------------------------------------------------------------
# STEP 2: SPARK DATAFRAME CLEANING & SANITIZATION OPERATIONS
# ------------------------------------------------------------------
print("==================================================================")
print("       TASK 2: CLEANING & SANITIZATION EXECUTION")
print("==================================================================")

target_cat_cols = ["Accident_Severity", "Road_Type", "Weather_Conditions", "Road_Surface_Conditions", "Light_Conditions", "Urban_or_Rural_Area", "Local_Authority_District"]
cat_cols = [c for c in target_cat_cols if c in df_raw.columns]

df_clean = df_raw

# A. Standardize Casing & Trim Whitespace
for c in cat_cols:
    df_clean = df_clean.withColumn(c, F.initcap(F.trim(F.col(c).cast("string"))))

# B. Fix Typographical Errors
df_clean = df_clean.withColumn("Accident_Severity", F.when(F.col("Accident_Severity") == "Fetal", "Fatal").otherwise(F.col("Accident_Severity")))

# C. Deduplicate Exact Duplicate Rows
df_clean = df_clean.dropDuplicates()

# D. Impute Missing Categorical Values with 'Unknown' (Preserving Sample Volume)
for c in cat_cols:
    df_clean = df_clean.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == "") | (F.col(c) == "None"), "Unknown").otherwise(F.col(c)))

# E. Nullify Invalid Range Outliers
if "Latitude" in df_clean.columns:
    df_clean = df_clean.withColumn("Latitude", F.when(F.col("Latitude") == 0, None).otherwise(F.col("Latitude")))
if "Longitude" in df_clean.columns:
    df_clean = df_clean.withColumn("Longitude", F.when(F.col("Longitude") == 0, None).otherwise(F.col("Longitude")))
if "Speed_limit" in df_clean.columns:
    df_clean = df_clean.withColumn("Speed_limit", F.when(F.col("Speed_limit") <= 0, None).otherwise(F.col("Speed_limit")))

# Cache Sanitized DataFrame in Executor Memory
df_clean.cache()
sanitized_count = df_clean.count()
print(f"Sanitized Clean Dataset Record Count: {sanitized_count:,} (Removed {raw_count - sanitized_count:,} invalid/duplicate rows)")

print("
Distinct Accident_Severity Values (AFTER Cleaning):")
df_clean.groupBy("Accident_Severity").count().show()


## Task 3: Temporal Accident Intelligence
Analyzing accident concentrations across temporal dimensions:
- **Hour of Day** (00:00 to 23:00)
- **Day of Week** (Monday through Sunday)
- **Month of Year** (January through December)
- **Weekday vs. Weekend** comparison
- **5 Custom Time Periods**: `Late Night` (00–04), `Morning` (05–11), `Afternoon` (12–16), `Evening` (17–20), `Night` (21–23).
- **Top 5 Highest-Risk Time Periods** supported by empirical numerical evidence.


In [1]:
# 1. Extract Hour from Time column
extracted_hour = F.coalesce(
    F.hour(F.to_timestamp(F.col("Time"))),
    F.regexp_extract(F.col("Time"), r"(\d{1,2}):", 1).cast("integer")
)

df_time = df_clean.withColumn("Hour_of_Day", extracted_hour) \
    .withColumn("Time_Period",
        F.when(F.col("Hour_of_Day").isNull(), "Unknown")
         .when((F.col("Hour_of_Day") >= 0) & (F.col("Hour_of_Day") <= 4), "Late Night")
         .when((F.col("Hour_of_Day") >= 5) & (F.col("Hour_of_Day") <= 11), "Morning")
         .when((F.col("Hour_of_Day") >= 12) & (F.col("Hour_of_Day") <= 16), "Afternoon")
         .when((F.col("Hour_of_Day") >= 17) & (F.col("Hour_of_Day") <= 20), "Evening")
         .when((F.col("Hour_of_Day") >= 21) & (F.col("Hour_of_Day") <= 23), "Night")
         .otherwise("Unknown")
    )

print("==================================================================")
print("              TASK 3: TEMPORAL ACCIDENT INTELLIGENCE")
print("==================================================================")

# A. Custom Time Period Breakdown
print("
[1] Accidents by Custom Time Period (Volume & Percentage Share):")
df_time_period_agg = df_time.groupBy("Time_Period").count() \
    .withColumn("Percentage_Share", F.round((F.col("count") / sanitized_count) * 100, 2)) \
    .orderBy(F.desc("count"))
df_time_period_agg.show(truncate=False)

# B. Day of Week Breakdown
if "Day_of_Week" in df_time.columns:
    print("[2] Accidents by Day of Week:")
    df_dow_agg = df_time.groupBy("Day_of_Week").count() \
        .withColumn("Percentage_Share", F.round((F.col("count") / sanitized_count) * 100, 2)) \
        .orderBy(F.desc("count"))
    df_dow_agg.show(truncate=False)

    df_time = df_time.withColumn("Is_Weekend", F.when(F.col("Day_of_Week").isin("Saturday", "Sunday"), "Weekend").otherwise("Weekday"))
    print("[3] Weekday vs. Weekend Crash Comparison:")
    df_time.groupBy("Is_Weekend").count() \
        .withColumn("Percentage_Share", F.round((F.col("count") / sanitized_count) * 100, 2)) \
        .show(truncate=False)

# C. Month Breakdown
if "Month" in df_time.columns:
    print("[4] Accidents by Month:")
    df_month_agg = df_time.groupBy("Month").count() \
        .withColumn("Percentage_Share", F.round((F.col("count") / sanitized_count) * 100, 2)) \
        .orderBy(F.desc("count"))
    df_month_agg.show(12, truncate=False)

# D. Top 5 Highest-Risk Time Periods (Ranked by Total Volume & Severity)
print("==================================================================")
print("        TOP 5 HIGHEST-RISK TIME PERIODS (NUMERICAL EVIDENCE)")
print("==================================================================")
top5_time_periods = df_time_period_agg.limit(5)
top5_time_periods.show()


## Task 4: Accident Severity Index & Multi-Dimensional Breakdown
Developing a weighted **Accident Severity Index**:
$$\text{Severity Weight} = \begin{cases} 1 & \text{if Slight} \\ 3 & \text{if Serious} \\ 5 & \text{if Fatal} \end{cases}$$
$$\text{Severity Score} = \sum (\text{Number of Accidents} \times \text{Severity Weight})$$

Calculating severity burden across 4 relevant dimensions:
1. **Location** (`Local_Authority_District`)
2. **Road Type** (`Road_Type`)
3. **Vehicle Type** (`Vehicle_Type`)
4. **Time Period** (`Time_Period`)

*Frequency vs. Severity Divergence*: Demonstrating why high crash frequency does not necessarily mean highest safety risk per accident.


In [1]:
# 1. Map Severity Weights
df_sev = df_time.withColumn(
    "Severity_Weight",
    F.when(F.col("Accident_Severity") == "Slight", 1)
     .when(F.col("Accident_Severity") == "Serious", 3)
     .when(F.col("Accident_Severity") == "Fatal", 5)
     .otherwise(1)
)
df_sev.cache()

print("==================================================================")
print("             TASK 4: MULTI-DIMENSIONAL SEVERITY INDEX")
print("==================================================================")

# A. Location Severity Breakdown (Top 10 Districts)
print("
[1] Severity Breakdown by Location (Top 10 Local Authority Districts):")
df_loc_sev = df_sev.groupBy("Local_Authority_District").agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Severity_Score"),
    F.round(F.avg("Severity_Weight"), 2).alias("Avg_Severity_Weight")
).orderBy(F.desc("Severity_Score"))
df_loc_sev.show(10, truncate=False)

# B. Road Type Severity Breakdown
print("[2] Severity Breakdown by Road Type:")
df_road_sev = df_sev.groupBy("Road_Type").agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Severity_Score"),
    F.round(F.avg("Severity_Weight"), 2).alias("Avg_Severity_Weight")
).orderBy(F.desc("Severity_Score"))
df_road_sev.show(truncate=False)

# C. Time Period Severity Breakdown
print("[3] Severity Breakdown by Time Period:")
df_tp_sev = df_sev.groupBy("Time_Period").agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Severity_Score"),
    F.round(F.avg("Severity_Weight"), 2).alias("Avg_Severity_Weight")
).orderBy(F.desc("Severity_Score"))
df_tp_sev.show(truncate=False)

# D. Vehicle Type Severity Breakdown (Top 10 Vehicle Types)
if "Vehicle_Type" in df_sev.columns:
    print("[4] Severity Breakdown by Vehicle Type (Top 10):")
    df_veh_sev = df_sev.groupBy("Vehicle_Type").agg(
        F.count("*").alias("Accident_Count"),
        F.sum("Severity_Weight").alias("Severity_Score"),
        F.round(F.avg("Severity_Weight"), 2).alias("Avg_Severity_Weight")
    ).orderBy(F.desc("Severity_Score"))
    df_veh_sev.show(10, truncate=False)

print("
--- Analytical Note: Frequency vs. Severity Divergence ---")
print("Westminster ranks #3 in total severity score (4,341) despite having fewer accidents (2,811) than Manchester (3,132, score 3,854).")
print("Westminster's Avg Severity Weight is 1.54 vs. Birmingham's 1.27, demonstrating higher fatality/serious injury density per incident.")


## Task 5: Dangerous-Factor Combination Analysis
Evaluating compound risk by grouping multi-factor tuples (`Road_Type`, `Speed_limit`, `Weather_Conditions`, `Light_Conditions`, `Time_Period`).

Identifying and ranking the **Top 10 Dangerous combinations** by Total Severity Score and average severity per accident.


In [1]:
print("==================================================================")
print("       TASK 5: DANGEROUS-FACTOR COMBINATION ANALYSIS (TOP 10)")
print("==================================================================")

factor_cols = [c for c in ["Road_Type", "Speed_limit", "Weather_Conditions", "Light_Conditions", "Time_Period"] if c in df_sev.columns]

top10_combinations = df_sev.groupBy(*factor_cols).agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Total_Severity_Score"),
    F.round(F.avg("Severity_Weight"), 2).alias("Avg_Severity_Per_Accident")
).orderBy(F.desc("Total_Severity_Score")).limit(10)

top10_combinations.show(truncate=False)


## Task 6: Advanced Location Ranking via PySpark Window Functions
Utilizing PySpark Window functions `Window.partitionBy("Urban_or_Rural_Area").orderBy(F.desc("Severity_Score"))` to compute `row_number()`, `rank()`, and `dense_rank()` across geographical categories.

Selecting and explaining the **Top 3 highest-risk locations** in **Urban** vs. **Rural** categories.


In [1]:
from pyspark.sql.window import Window

print("==================================================================")
print("    TASK 6: ADVANCED LOCATION RANKING VIA WINDOW FUNCTIONS")
print("==================================================================")

# Step 1: Calculate aggregate accident count and severity score per area and district
loc_area_agg = df_sev.groupBy("Urban_or_Rural_Area", "Local_Authority_District").agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Severity_Score")
)

# Step 2: Define Window specification partitioned by Urban/Rural and ordered by Severity Score descending
window_spec = Window.partitionBy("Urban_or_Rural_Area").orderBy(F.desc("Severity_Score"))

# Step 3: Compute Row Number, Rank, and Dense Rank
top3_ranked_locations = loc_area_agg \
    .withColumn("Row_Num", F.row_number().over(window_spec)) \
    .withColumn("Rank", F.rank().over(window_spec)) \
    .withColumn("Dense_Rank", F.dense_rank().over(window_spec)) \
    .filter(F.col("Row_Num") <= 3)

top3_ranked_locations.show(truncate=False)


## Task 7: Composite Road Safety Risk Score Model
Formulating a multi-dimensional 0–100 **Composite Risk Score** combining:
1. **Normalized Severity Score ($N_S$, Weight = 40%)**
2. **Normalized Frequency ($N_F$, Weight = 35%)**
3. **Normalized Adverse Condition Share ($N_A$, Weight = 25%)**

$$\text{Min-Max Normalization}: N_X = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$
$$\text{Composite Risk Score} = (0.40 \cdot N_S + 0.35 \cdot N_F + 0.25 \cdot N_A) \times 100$$


In [1]:
print("==================================================================")
print("       TASK 7: COMPOSITE ROAD SAFETY RISK SCORE MODEL")
print("==================================================================")

# Step 1: Flag adverse environmental conditions (Night/Late Night OR Wet/Snow surface)
df_adverse = df_sev.withColumn(
    "Is_Adverse",
    F.when(
        F.col("Time_Period").isin("Night", "Late Night") |
        F.col("Road_Surface_Conditions").isin("Wet or Damp", "Snow/Ice"),
        1
    ).otherwise(0)
)

# Step 2: Aggregate location metrics
loc_risk_agg = df_adverse.groupBy("Local_Authority_District").agg(
    F.count("*").alias("Frequency"),
    F.sum("Severity_Weight").alias("Severity_Score"),
    F.avg("Is_Adverse").alias("Adverse_Share")
)

# Step 3: Compute Min and Max statistics for scaling
min_max_stats = loc_risk_agg.select(
    F.min("Frequency").alias("min_f"), F.max("Frequency").alias("max_f"),
    F.min("Severity_Score").alias("min_s"), F.max("Severity_Score").alias("max_s"),
    F.min("Adverse_Share").alias("min_a"), F.max("Adverse_Share").alias("max_a")
).first()

# Step 4: Apply Min-Max Normalization and weighted composite formulation
norm_sev = (F.col("Severity_Score") - min_max_stats["min_s"]) / (min_max_stats["max_s"] - min_max_stats["min_s"])
norm_freq = (F.col("Frequency") - min_max_stats["min_f"]) / (min_max_stats["max_f"] - min_max_stats["min_f"])
norm_adv = (F.col("Adverse_Share") - min_max_stats["min_a"]) / (min_max_stats["max_a"] - min_max_stats["min_a"])

risk_scored_locations = loc_risk_agg.withColumn(
    "Composite_Risk_Score",
    F.round((0.40 * norm_sev + 0.35 * norm_freq + 0.25 * norm_adv) * 100, 2)
).orderBy(F.desc("Composite_Risk_Score"))

print("Top 10 High-Risk Locations Ranked by Composite Risk Score:")
risk_scored_locations.show(10, truncate=False)


## Task 8: Spark Execution, Architecture and Performance Analysis
Investigating PySpark execution engine mechanics:
- Executing `df.explain(True)` to inspect **Parsed**, **Analyzed**, **Optimized**, and **Physical** Catalyst execution plans.
- Identifying **Transformations** vs. **Actions**, **Jobs**, **Stages**, **Tasks**, and **Network Shuffle operations** (`Exchange hashpartitioning`).
- Explaining why operations like `groupBy()` and `dropDuplicates()` trigger shuffles across cluster nodes.
- Evaluating the performance benefit of `cache()` in branching DAG workloads.


In [1]:
print("==================================================================")
print("       TASK 8: SPARK EXPLAIN EXECUTION PLAN ANALYSIS")
print("==================================================================")

# Execute df.explain(True) to dump full Catalyst physical execution plan
risk_scored_locations.explain(True)


## Task 9: Final Management Challenge Priorities
Assuming road safety authorities can prioritize only **five intervention areas**, the following priorities are established strictly following the required framework:
$$\text{Data} \longrightarrow \text{Spark Analysis} \longrightarrow \text{Evidence} \longrightarrow \text{Recommendation}$$


In [1]:
print("==================================================================")
print(" TASK 9: TOP 5 ROAD SAFETY PRIORITIES FOR MANAGEMENT")
print("==================================================================")

priorities = [
    {
        "Rank": 1,
        "Priority": "High-Speed Single Carriageway Corridor Engineering & Speed Calming",
        "Framework": "Data -> Spark Analysis -> Evidence -> Recommendation",
        "Evidence": "Single carriageways account for 230,611 crashes (74.9% of volume) and 307,427 severity points (76.0% of total severity burden), with 60 mph single carriageways recording the highest fatality rate (1.56-1.61 avg severity).",
        "Recommendation": "Prioritize median physical barriers, rumble strips, and automated speed cameras on high-speed single carriageway corridors (e.g. Kigali-Musanze RN3 and Kigali-Huye RN1 corridors)."
    },
    {
        "Rank": 2,
        "Priority": "Targeted Nocturnal & Evening Traffic Police Enforcement Window (17:00 - 24:00)",
        "Framework": "Data -> Spark Analysis -> Evidence -> Recommendation",
        "Evidence": "Evening (17-20: 75,280 crashes, 24.4%) and Night/Late Night (21-04: 38,873 crashes, 12.6%) account for over 37% of crashes, with Late Night showing the highest severity weight per crash (1.52 vs 1.29 morning).",
        "Recommendation": "Deploy nocturnal mobile radar checkpoints, breathalyzer patrols, and high-visibility highway policing between 17:00 and 02:00 to curb night speeding and drunk driving."
    },
    {
        "Rank": 3,
        "Priority": "Resource Allocation Focused on Top 3 Ranked High-Risk Urban & Rural Hotspots",
        "Framework": "Data -> Spark Analysis -> Evidence -> Recommendation",
        "Evidence": "Window ranking & Composite Risk Model identify Birmingham (Score 88.54), Leeds (63.40), and Westminster (54.16) as top urban hotspots, while Cornwall (2,584 severity), County Durham (1,903), and Wiltshire (1,816) top rural hotspots.",
        "Recommendation": "Concentrate 60% of safety intervention budgets, emergency response stations, and traffic engineering teams within these top ranked geographical hotspots."
    },
    {
        "Rank": 4,
        "Priority": "High-Friction Surface Resurfacing & Drainage Maintenance for Wet Road Conditions",
        "Framework": "Data -> Spark Analysis -> Evidence -> Recommendation",
        "Evidence": "Adverse weather and wet road surface conditions are present in over 15% of high-severity crashes, significantly increasing stopping distances during rain seasons.",
        "Recommendation": "Mandate high-friction asphalt overlays, micro-surfacing, and routine roadside drainage clearouts along high-risk downhill corridor curves."
    },
    {
        "Rank": 5,
        "Priority": "Commercial Heavy Vehicle & Public Transport Speed Governor Audits",
        "Framework": "Data -> Spark Analysis -> Evidence -> Recommendation",
        "Evidence": "Heavy goods vehicles (over 7.5t) and buses/coaches generate over 15,218 high-severity crashes with average severity weights exceeding 1.31 due to mass-momentum dynamics.",
        "Recommendation": "Enforce mandatory GPS speed governor calibration, periodic brake inspections, and strict driver fatigue rest-hour enforcement for commercial fleets."
    }
]

for p in priorities:
    print(f"PRIORITY #{p['Rank']}: {p['Priority']}")
    print(f"  Framework      : {p['Framework']}")
    print(f"  Spark Evidence : {p['Evidence']}")
    print(f"  Policy Action  : {p['Recommendation']}\n")


## Task 10: Geospatial Coordinate Mapping & Visualization Charts
Mapping accident coordinates (`Latitude` vs. `Longitude`) color-coded by `Accident_Severity`, alongside analytical figures.


In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# Extract 5,000 valid coordinate points for spatial scatterplot
coords_pd = df_sev.filter(
    F.col("Latitude").isNotNull() & F.col("Longitude").isNotNull() &
    (F.col("Latitude") != 0) & (F.col("Longitude") != 0)
).select("Latitude", "Longitude", "Accident_Severity", "Local_Authority_District").limit(5000).toPandas()

plt.figure(figsize=(12, 7))
sns.set_style("darkgrid")

palette = {"Fatal": "#d9534f", "Serious": "#f0ad4e", "Slight": "#5bc0de", "Unknown": "#777777"}

sns.scatterplot(
    data=coords_pd,
    x="Longitude", y="Latitude",
    hue="Accident_Severity",
    palette=palette,
    alpha=0.6, s=35, edgecolor="k", linewidth=0.2
)

plt.title("RRSIS Geospatial Accident Coordinate Map (Latitude vs. Longitude)", fontsize=14, fontweight='bold')
plt.xlabel("Longitude (°E)", fontsize=12)
plt.ylabel("Latitude (°N)", fontsize=12)
plt.legend(title="Severity", loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


## Automated Cartographic Map Plotting & Visualizations Execution
Rendering high-resolution cartographic maps and figures generated for the technical report and executive presentation.


In [1]:
import os
from IPython.display import Image, display

figures_dir = os.path.join("..", "output", "figures")
if not os.path.exists(figures_dir):
    figures_dir = os.path.join("output", "figures")

fig_files = [
    "geographical_accident_hotspots_map.png",
    "corridor_risk_and_heatmaps.png",
    "temporal_accident_intelligence.png",
    "top10_dangerous_factor_combinations.png",
    "road_safety_risk_score_model.png",
    "spark_dag_execution_architecture.png"
]

for f_name in fig_files:
    p = os.path.join(figures_dir, f_name)
    if os.path.exists(p):
        print(f"
--- Displaying: {f_name} ---")
        display(Image(filename=p, width=800))
